In [ ]:
Modification:
The timestamp column (2026-03-10T08:02:48.202Z) is parsed as UTC ISO8601 and the date is extracted from it. 
The ts column handles both formats automatically — 10-03-26 13:30 (DD-MM-YY HH:MM) and 7:08:27 PM (12-hour with AM/PM). 

2. Fixed Y-axis limits
All four plot types now have explicit, data-independent Y limits defined in the STYLE block at the top:
ymax_full_raw    = 1800   # linear
ymax_per_day     = 1800   # linear
ymax_hourly      = 400
ymax_halfhourly  = 400
ymin_log         = 1      # log lower bound
ymax_log_full_raw = 1800  # log upper bound
ymax_log_per_day  = 1800

3. Log-scale plots
full_raw_log and per_day_log sub-folders are generated automatically alongside the linear ones. 

4. X-axis for 20–25 days
Full-raw and resampled plots now tick every 2 days

5. Full customisation in one place
Every font size, weight, tick length, legend position, rotation, grid opacity, line width, 
and Y limit is in the STYLE dict at the top — nothing is hardcoded inside the plot functions.

#### 1.A.

In [11]:
"""
Air Quality Time Series Plotting Script
========================================
Generates plots (JPG) from an Excel file with 9 sheets:
  S-series: S1_Dharavi, S2_AntopHill, S3_CAAQMS, S4_Chembur
  C-series: C1_Marol, C2_KurlaWest, C3_KhairaniRd, C4_Ghatkopar, C5_Powai

Plot categories:
  A) Full raw time series          → linear Y + log Y  (2 pollutants × 2 series × 2 scales)
  B) Per-day time series           → linear Y + log Y  (days × 2 pollutants × 2 series × 2 scales)
  C) Hourly averaged time series   → linear Y           (2 pollutants × 2 series)
  D) Half-hourly averaged t-series → linear Y           (2 pollutants × 2 series)

Output folder structure:
  OUTPUT_DIR/
    S_series/
      full_raw/          ← linear
      full_raw_log/      ← log Y
      per_day/           ← linear
      per_day_log/       ← log Y
      hourly/
      halfhourly/
    C_series/
      (same sub-folders)

Date source  : 'timestamp' column  e.g. "2026-03-10T08:02:48.202Z"
Time source  : 'ts' column         e.g. "10-03-26 13:30"  OR  "7:08:27 PM"
"""

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, LogLocator, LogFormatter

# =============================================================================
# USER CONFIGURATION — edit paths here
# =============================================================================
INPUT_FILE = r"D:/Hyperlocal_Study/Data_Analysis/Sample_Data_Mar/30_Mar_Data_LAMP.xlsx"
OUTPUT_DIR = r"C:/Users/Atique/Hyperlocal_LAMP/Visualisation/Data_Quality_Assessment/Mar30/"
# =============================================================================

# ── Sheet groups ──────────────────────────────────────────────────────────────
S_SHEETS = ["S1_Dharavi", "S2_AntopHill", "S3_CAAQMS", "S4_Chembur", "S5_KurlaEast"]
C_SHEETS = ["C1_Marol", "C2_KurlaWest", "C3_KhairaniRd", "C4_Ghatkopar", "C5_Powai"]

POLLUTANTS = {
    "PM2.5": "pm2.5",
    "PM10":  "pm10",
}

# ── Colour palette (one colour per sheet) ─────────────────────────────────────
COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22",
]

# =============================================================================
# STYLE CONFIGURATION — customise everything here
# Numbers stay the same as the original; change any value freely.
# =============================================================================
STYLE = dict(

    # ── Figure ────────────────────────────────────────────────────────────────
    fig_width           = 16,
    fig_height          = 5,
    dpi                 = 150,
    jpg_quality         = 95,

    # ── Lines ─────────────────────────────────────────────────────────────────
    linewidth           = 1.2,

    # ── Axis labels ───────────────────────────────────────────────────────────
    xlabel_size         = 13,
    xlabel_weight       = "normal",       # "normal" or "bold"
    ylabel_size         = 13,
    ylabel_weight       = "normal",

    # ── Tick labels ───────────────────────────────────────────────────────────
    xtick_labelsize     = 12,
    ytick_labelsize     = 12,
    xtick_rotation      = 30,

    # ── Tick marks ────────────────────────────────────────────────────────────
    tick_major_len      = 6,
    tick_minor_len      = 3,
    tick_width          = 1.0,

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_fontsize     = 12,
    legend_title_size   = 12,
    legend_title        = "Station",      # label above legend entries
    legend_framealpha   = 0.85,
    legend_edgecolor    = "grey",
    legend_loc          = "upper right",  # e.g. "upper left", "lower right"

    # ── Grid ──────────────────────────────────────────────────────────────────
    grid_alpha          = 0.3,
    grid_lw             = 0.7,

    # ── Spines ────────────────────────────────────────────────────────────────
    spine_lw            = 1.1,

    # ── Y-axis fixed limits (linear scale) ────────────────────────────────────
    ymax_full_raw       = 1800,   # full raw plots
    ymax_per_day        = 1800,   # per-day plots
    ymax_hourly         = 400,    # hourly resampled plots
    ymax_halfhourly     = 400,    # half-hourly resampled plots
    ymin_linear         = 0,      # bottom of linear Y axis

    # ── Y-axis limits (log scale) ─────────────────────────────────────────────
    ymin_log            = 1,      # log Y lower bound (must be > 0)
    ymax_log_full_raw   = 1800,   # log Y upper bound for full_raw_log
    ymax_log_per_day    = 1800,   # log Y upper bound for per_day_log
)


# =============================================================================
# TIMESTAMP PARSING
# =============================================================================

def parse_timestamp_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'timestamp' column.
    Expected format: "2026-03-10T08:02:48.202Z"
    Returns a tz-naive datetime Series (date + time both extracted).
    """
    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return dt.dt.tz_convert(None)


def parse_ts_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'ts' column. Handles all observed formats:

      Already parsed by pandas (most common when reading Excel):
        Timestamp('2026-03-12 09:58:48') or datetime.datetime(2026, 3, 14, 19, 8, 27)
        → .time() is called directly, no string parsing needed.

      String fallback:
        "10-03-26 13:30"  DD-MM-YY HH:MM
        "7:08:27 PM"      12-hour AM/PM
        "19:08:27"        24-hour time-only

    Returns a Series of datetime.time objects.
    """
    import datetime as _dt

    def _parse_one(val):
        if val is None:
            return pd.NaT
        try:
            if pd.isna(val):
                return pd.NaT
        except (TypeError, ValueError):
            pass

        # Already a datetime / Timestamp — just extract time
        if isinstance(val, (pd.Timestamp, _dt.datetime)):
            return val.time()
        if isinstance(val, _dt.time):
            return val

        # String fallback
        s = str(val).strip()
        if re.match(r"^\d{2}-\d{2}-\d{2}\s+\d{1,2}:\d{2}", s):
            try:
                return pd.to_datetime(s, format="%d-%m-%y %H:%M").time()
            except Exception:
                pass
        for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
            try:
                return pd.to_datetime(s, format=fmt).time()
            except Exception:
                continue
        return pd.NaT

    return series.apply(_parse_one)


# =============================================================================
# DATA LOADING
# =============================================================================

def load_data(filepath: str, sheets: list) -> dict:
    """
    Load & clean data from specified sheets.

    Datetime used for plotting is constructed as:
      date  from 'timestamp' column  (ISO8601 UTC string)
      time  from 'ts' column         (DD-MM-YY HH:MM  OR  H:MM:SS AM/PM)

    The combined datetime is stored in column 'plot_dt'.
    """
    dfs = {}
    for sheet in sheets:
        df = pd.read_excel(filepath, sheet_name=sheet)
        df.columns = df.columns.str.strip().str.lower()

        # ── Extract date from 'timestamp' ────────────────────────────────────
        if "timestamp" not in df.columns:
            raise KeyError(f"Sheet '{sheet}' has no 'timestamp' column. "
                           f"Found: {df.columns.tolist()}")
        full_dt = parse_timestamp_col(df["timestamp"])
        date_part = full_dt.dt.date   # datetime.date objects

        # ── Extract time from 'ts' ────────────────────────────────────────────
        if "ts" not in df.columns:
            raise KeyError(f"Sheet '{sheet}' has no 'ts' column. "
                           f"Found: {df.columns.tolist()}")
        time_part = parse_ts_col(df["ts"])   # datetime.time objects

        # ── Combine date + time into a single plot_dt column ──────────────────
        def _combine(row):
            d, t = row["_date"], row["_time"]
            if pd.isna(d) or pd.isna(t) or t is pd.NaT:
                return pd.NaT
            try:
                return pd.Timestamp.combine(d, t)
            except Exception:
                return pd.NaT

        df["_date"] = date_part
        df["_time"] = time_part
        df["plot_dt"] = df.apply(_combine, axis=1)
        df = df.drop(columns=["_date", "_time"])

        df = df.sort_values("plot_dt").reset_index(drop=True)

        # Keep only the columns we need
        poll_cols = [v for v in POLLUTANTS.values() if v in df.columns]
        df = df[["plot_dt"] + poll_cols]
        dfs[sheet] = df

    return dfs


# =============================================================================
# HELPERS
# =============================================================================

def apply_style(ax, xlabel="Date & Time", ylabel="Concentration (µg/m³)"):
    s = STYLE
    ax.set_xlabel(xlabel, fontsize=s["xlabel_size"],
                  fontweight=s["xlabel_weight"], labelpad=8)
    ax.set_ylabel(ylabel, fontsize=s["ylabel_size"],
                  fontweight=s["ylabel_weight"], labelpad=8)
    ax.tick_params(axis="both", which="major",
                   labelsize=s["xtick_labelsize"],
                   length=s["tick_major_len"], width=s["tick_width"])
    ax.tick_params(axis="y", which="major",
                   labelsize=s["ytick_labelsize"])
    ax.tick_params(axis="both", which="minor",
                   length=s["tick_minor_len"], width=s["tick_width"])
    ax.grid(True, which="major", linestyle="--",
            alpha=s["grid_alpha"], linewidth=s["grid_lw"])
    ax.grid(True, which="minor", linestyle=":",
            alpha=s["grid_alpha"] * 0.6, linewidth=s["grid_lw"] * 0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(s["spine_lw"])


def apply_log_y(ax):
    """Switch Y axis to log scale with sensible ticks."""
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10, numticks=10))
    ax.yaxis.set_major_formatter(LogFormatter(base=10, labelOnlyBase=False))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs="auto", numticks=10))


def set_linear_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin_linear"], ymax)
    ax.yaxis.set_minor_locator(AutoMinorLocator())


def set_log_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin_log"], ymax)


def format_xaxis_datetime(ax, locator, formatter):
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    plt.setp(ax.get_xticklabels(),
             rotation=STYLE["xtick_rotation"], ha="right",
             fontsize=STYLE["xtick_labelsize"])


def add_legend(ax):
    s = STYLE
    ax.legend(
        title=s["legend_title"],
        fontsize=s["legend_fontsize"],
        title_fontsize=s["legend_title_size"],
        framealpha=s["legend_framealpha"],
        edgecolor=s["legend_edgecolor"],
        loc=s["legend_loc"],
    )


def save_fig(fig, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=STYLE["dpi"], bbox_inches="tight",
                format="jpeg",
                pil_kwargs={"quality": STYLE["jpg_quality"]})
    plt.close(fig)
    print(f"  Saved → {os.path.basename(path)}")


def make_figure():
    return plt.subplots(figsize=(STYLE["fig_width"], STYLE["fig_height"]))


# =============================================================================
# PLOT FUNCTIONS
# =============================================================================

# ── A) Full raw time series (linear or log) ───────────────────────────────────

def plot_full_timeseries(dfs: dict, poll_col: str, out_path: str,
                         log_scale: bool = False):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        ax.plot(df["plot_dt"], df[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    # X axis: data spans 20-25 days → tick every 2 days
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)

    if log_scale:
        apply_log_y(ax)
        set_log_ylim(ax, STYLE["ymax_log_full_raw"])
    else:
        set_linear_ylim(ax, STYLE["ymax_full_raw"])

    fig.tight_layout()
    save_fig(fig, out_path)


# ── B) Per-day time series (linear or log) ────────────────────────────────────

def plot_daily_timeseries(dfs: dict, poll_col: str, day: pd.Timestamp,
                          out_path: str, log_scale: bool = False):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        mask = df["plot_dt"].dt.date == day.date()
        day_df = df[mask]
        if day_df.empty:
            continue
        ax.plot(day_df["plot_dt"], day_df[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax, xlabel=f"Time on {day.strftime('%d %b %Y')}")
    format_xaxis_datetime(
        ax,
        mdates.HourLocator(interval=2),
        mdates.DateFormatter("%H:%M"),
    )
    add_legend(ax)

    if log_scale:
        apply_log_y(ax)
        set_log_ylim(ax, STYLE["ymax_log_per_day"])
    else:
        set_linear_ylim(ax, STYLE["ymax_per_day"])

    fig.tight_layout()
    save_fig(fig, out_path)


# ── C & D) Resampled time series (hourly / half-hourly, linear only) ──────────

def plot_resampled_timeseries(dfs: dict, poll_col: str,
                              resample_rule: str, out_path: str,
                              ymax: int):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        resampled = (
            df.set_index("plot_dt")[poll_col]
              .resample(resample_rule)
              .mean()
              .dropna()
              .reset_index()
        )
        ax.plot(resampled["plot_dt"], resampled[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)
    set_linear_ylim(ax, ymax)
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    fig.tight_layout()
    save_fig(fig, out_path)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading data …")
    s_dfs = load_data(INPUT_FILE, S_SHEETS)
    c_dfs = load_data(INPUT_FILE, C_SHEETS)

    series_map = {"S": s_dfs, "C": c_dfs}

    # ── Diagnostics: show raw column samples and parse results ──────────────
    print("\n── PARSE DIAGNOSTICS ──────────────────────────────────────────────────")
    for series_tag, dfs in series_map.items():
        for sheet, df in dfs.items():
            raw = pd.read_excel(INPUT_FILE, sheet_name=sheet)
            raw.columns = raw.columns.str.strip().str.lower()

            ts_raw   = raw["timestamp"].iloc[0] if "timestamp" in raw.columns else "MISSING COLUMN"
            tscol    = raw["ts"].iloc[0]         if "ts"        in raw.columns else "MISSING COLUMN"
            n_valid  = df["plot_dt"].dropna().shape[0]
            n_total  = df.shape[0]
            sample   = df["plot_dt"].dropna().iloc[0] if n_valid > 0 else "ALL NaT — parse failed"

            print(f"  [{sheet}]")
            print(f"    timestamp[0] raw  : {repr(ts_raw)}")
            print(f"    ts[0]        raw  : {repr(tscol)}")
            print(f"    plot_dt[0] parsed : {sample}")
            print(f"    valid rows        : {n_valid} / {n_total}")
            if n_valid == 0:
                print(f"    *** WARNING: 0 rows parsed — check format above ***")
    print("───────────────────────────────────────────────────────────────────────\n")

    # Derive date range from the loaded data itself
    all_dates_set = set()
    for dfs in series_map.values():
        for df in dfs.values():
            all_dates_set.update(df["plot_dt"].dropna().dt.date.unique())
    all_dates = sorted(all_dates_set)

    if not all_dates:
        print("\n❌  ERROR: No valid dates found across any sheet.")
        print("    Check the PARSE DIAGNOSTICS output above.")
        print("    Common causes:")
        print("      • Column not named exactly 'timestamp' or 'ts' in your Excel")
        print("      • 'timestamp' format is not ISO8601 e.g. '2026-03-10T08:02:48.202Z'")
        print("      • 'ts' format is not '10-03-26 13:30' or '7:08:27 PM'")
        print("      • Paste the actual first-row values shown above and share them")
        return

    print(f"  Date range in data: {all_dates[0]} → {all_dates[-1]} "
          f"({len(all_dates)} days)")

    resample_map = {
        "hourly":     ("1h",    STYLE["ymax_hourly"]),
        "halfhourly": ("30min", STYLE["ymax_halfhourly"]),
    }

    total = 0

    for poll_label, poll_col in POLLUTANTS.items():
        poll_tag = poll_label.replace(".", "")  # PM25 / PM10

        for series_tag, dfs in series_map.items():
            base = os.path.join(OUTPUT_DIR, series_tag + "_series")

            # ── A) Full raw — linear ──────────────────────────────────────────
            fname = f"{series_tag}_{poll_tag}_full_raw.jpg"
            out   = os.path.join(base, "full_raw", fname)
            print(f"\n[A-linear] Full raw | {series_tag} | {poll_label}")
            plot_full_timeseries(dfs, poll_col, out, log_scale=False)
            total += 1

            # ── A) Full raw — log ─────────────────────────────────────────────
            fname_log = f"{series_tag}_{poll_tag}_full_raw_log.jpg"
            out_log   = os.path.join(base, "full_raw_log", fname_log)
            print(f"[A-log]    Full raw log | {series_tag} | {poll_label}")
            plot_full_timeseries(dfs, poll_col, out_log, log_scale=True)
            total += 1

            # ── B) Per-day — linear & log ─────────────────────────────────────
            for day in all_dates:
                day_ts  = pd.Timestamp(day)
                day_str = day_ts.strftime("%Y-%m-%d")

                # linear
                fname = f"{series_tag}_{poll_tag}_{day_str}.jpg"
                out   = os.path.join(base, "per_day", fname)
                print(f"[B-linear] Per-day | {series_tag} | {poll_label} | {day_str}")
                plot_daily_timeseries(dfs, poll_col, day_ts, out, log_scale=False)
                total += 1

                # log
                fname_log = f"{series_tag}_{poll_tag}_{day_str}_log.jpg"
                out_log   = os.path.join(base, "per_day_log", fname_log)
                print(f"[B-log]    Per-day log | {series_tag} | {poll_label} | {day_str}")
                plot_daily_timeseries(dfs, poll_col, day_ts, out_log, log_scale=True)
                total += 1

            # ── C & D) Resampled ──────────────────────────────────────────────
            for res_label, (res_rule, ymax) in resample_map.items():
                fname = f"{series_tag}_{poll_tag}_{res_label}.jpg"
                out   = os.path.join(base, res_label, fname)
                print(f"[C/D] {res_label:12s} | {series_tag} | {poll_label}")
                plot_resampled_timeseries(dfs, poll_col, res_rule, out, ymax)
                total += 1

    print(f"\n✅  Done — {total} plots saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Loading data …

── PARSE DIAGNOSTICS ──────────────────────────────────────────────────
  [S1_Dharavi]
    timestamp[0] raw  : '2026-03-12T04:30:47.046Z'
    ts[0]        raw  : Timestamp('2026-03-12 09:58:48')
    plot_dt[0] parsed : 2026-03-12 00:00:28
    valid rows        : 47095 / 47095
  [S2_AntopHill]
    timestamp[0] raw  : '2026-03-05T07:15:13.701Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:41:43')
    plot_dt[0] parsed : 2026-03-05 00:00:26
    valid rows        : 65116 / 65116
  [S3_CAAQMS]
    timestamp[0] raw  : '2026-03-05T07:15:09.425Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:43:23')
    plot_dt[0] parsed : 2026-03-05 00:00:13
    valid rows        : 53716 / 53716
  [S4_Chembur]
    timestamp[0] raw  : '2026-03-05T07:14:56.479Z'
    ts[0]        raw  : Timestamp('2026-01-29 07:54:19')
    plot_dt[0] parsed : 2026-03-05 00:00:16
    valid rows        : 71307 / 71307
  [C1_Marol]
    timestamp[0] raw  : '2026-03-12T04:33:16.108Z'
    ts[0]        raw  : dat

In [14]:
"""
resave_plots.py
===============
Copies all JPG plots from the old output folder to the new correct folder,
preserving the exact sub-folder structure.

Run this once after the main plotting script has already completed.
No data loading or re-plotting — purely a file copy operation.
"""

import os
import shutil

# =============================================================================
# EDIT THESE TWO PATHS ONLY
# =============================================================================
OLD_DIR = r"C:/Users/Atique/Hyperlocal_LAMP/Visualisation/Data_Quality_Assessment/Mar30"
NEW_DIR = r"C:/Users/Atique/LAMP_Coding/Visualisation/Data_Quality_Assessment/Mar30"
# =============================================================================

def resave():
    if not os.path.exists(OLD_DIR):
        print(f"❌  Source folder not found:\n    {OLD_DIR}")
        return

    copied  = 0
    skipped = 0
    errors  = 0

    for root, dirs, files in os.walk(OLD_DIR):
        jpg_files = [f for f in files if f.lower().endswith(".jpg")]
        if not jpg_files:
            continue

        # Mirror the sub-folder structure under NEW_DIR
        rel_path  = os.path.relpath(root, OLD_DIR)
        dest_dir  = os.path.join(NEW_DIR, rel_path)
        os.makedirs(dest_dir, exist_ok=True)

        for fname in jpg_files:
            src  = os.path.join(root, fname)
            dest = os.path.join(dest_dir, fname)

            if os.path.exists(dest):
                skipped += 1
                continue
            try:
                shutil.copy2(src, dest)   # copy2 preserves metadata
                copied += 1
            except Exception as e:
                print(f"  ⚠  Failed to copy {fname}: {e}")
                errors += 1

    print(f"\n✅  Done.")
    print(f"    Copied  : {copied} files")
    print(f"    Skipped : {skipped} files (already existed at destination)")
    print(f"    Errors  : {errors}")
    print(f"\n    Destination: {NEW_DIR}")


if __name__ == "__main__":
    resave()


✅  Done.
    Copied  : 224 files
    Skipped : 0 files (already existed at destination)
    Errors  : 0

    Destination: C:/Users/Atique/LAMP_Coding/Visualisation/Data_Quality_Assessment/Mar30


#### 1.B.

In [16]:
"""
Air Quality Time Series Plotting Script
========================================
Generates plots (JPG) from an Excel file with 9 sheets:
  S-series: S1_Dharavi, S2_AntopHill, S3_CAAQMS, S4_Chembur
  C-series: C1_Marol, C2_KurlaWest, C3_KhairaniRd, C4_Ghatkopar, C5_Powai

Plot categories:
  A) Full raw time series          → linear Y + log Y  (2 pollutants × 2 series × 2 scales)
  B) Per-day time series           → linear Y + log Y  (days × 2 pollutants × 2 series × 2 scales)
  C) Hourly averaged time series   → linear Y           (2 pollutants × 2 series)
  D) Half-hourly averaged t-series → linear Y           (2 pollutants × 2 series)

Output folder structure:
  OUTPUT_DIR/
    S_series/
      full_raw/          ← linear
      full_raw_log/      ← log Y
      per_day/           ← linear
      per_day_log/       ← log Y
      hourly/
      halfhourly/
    C_series/
      (same sub-folders)

Date source  : 'timestamp' column  e.g. "2026-03-10T08:02:48.202Z"
Time source  : 'ts' column         e.g. "10-03-26 13:30"  OR  "7:08:27 PM"
"""

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, LogLocator, LogFormatter

# =============================================================================
# USER CONFIGURATION — edit paths here
# =============================================================================
INPUT_FILE = r"D:/Hyperlocal_Study/Data_Analysis/Sample_Data_Mar/30_Mar_Data_LAMP.xlsx"
OUTPUT_DIR = r"C:/Users/Atique/LAMP_Coding/Visualisation/Data_Quality_Assessment/Mar30/"
# =============================================================================

# ── Sheet groups ──────────────────────────────────────────────────────────────
S_SHEETS = ["S1_Dharavi", "S2_AntopHill", "S3_CAAQMS", "S4_Chembur","S5_KurlaEast"]
C_SHEETS = ["C1_Marol", "C2_KurlaWest", "C3_KhairaniRd", "C4_Ghatkopar", "C5_Powai"]

POLLUTANTS = {
    "PM2.5": "pm2.5",
    "PM10":  "pm10",
}

# ── Colour palette (one colour per sheet) ─────────────────────────────────────
COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22",
]

# =============================================================================
# STYLE CONFIGURATION — customise everything here
# Numbers stay the same as the original; change any value freely.
# =============================================================================
STYLE = dict(

    # ── Figure ────────────────────────────────────────────────────────────────
    fig_width           = 16,
    fig_height          = 5,
    dpi                 = 150,
    jpg_quality         = 95,

    # ── Lines ─────────────────────────────────────────────────────────────────
    linewidth           = 1.2,

    # ── Axis labels ───────────────────────────────────────────────────────────
    xlabel_size         = 13,
    xlabel_weight       = "normal",       # "normal" or "bold"
    ylabel_size         = 13,
    ylabel_weight       = "normal",

    # ── Tick labels ───────────────────────────────────────────────────────────
    xtick_labelsize     = 12,
    ytick_labelsize     = 12,
    xtick_rotation      = 30,

    # ── Tick marks ────────────────────────────────────────────────────────────
    tick_major_len      = 6,
    tick_minor_len      = 3,
    tick_width          = 1.0,

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_fontsize     = 12,
    legend_title_size   = 12,
    legend_title        = "Station",      # label above legend entries
    legend_framealpha   = 0.85,
    legend_edgecolor    = "grey",
    legend_loc          = "upper right",  # e.g. "upper left", "lower right"

    # ── Grid ──────────────────────────────────────────────────────────────────
    grid_alpha          = 0.3,
    grid_lw             = 0.7,

    # ── Spines ────────────────────────────────────────────────────────────────
    spine_lw            = 1.1,

    # ── Y-axis fixed limits (linear scale) ────────────────────────────────────
    # full_raw and per_day: separate limits per pollutant
    ymax_full_raw_pm25  = 1000,   # full raw  — PM2.5
    ymax_full_raw_pm10  = 1800,   # full raw  — PM10
    ymax_per_day_pm25   = 1000,   # per-day   — PM2.5
    ymax_per_day_pm10   = 1800,   # per-day   — PM10
    # hourly / half-hourly: single limit (both pollutants)
    ymax_hourly         = 400,    # hourly resampled plots
    ymax_halfhourly     = 400,    # half-hourly resampled plots
    ymin_linear         = 0,      # bottom of linear Y axis

    # ── Y-axis limits (log scale) ─────────────────────────────────────────────
    ymin_log               = 1,      # log Y lower bound (must be > 0)
    ymax_log_full_raw_pm25 = 1000,   # log Y upper bound for full_raw_log — PM2.5
    ymax_log_full_raw_pm10 = 1800,   # log Y upper bound for full_raw_log — PM10
    ymax_log_per_day_pm25  = 1000,   # log Y upper bound for per_day_log  — PM2.5
    ymax_log_per_day_pm10  = 1800,   # log Y upper bound for per_day_log  — PM10
)


# =============================================================================
# TIMESTAMP PARSING
# =============================================================================

def parse_timestamp_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'timestamp' column.
    Expected format: "2026-03-10T08:02:48.202Z"
    Returns a tz-naive datetime Series (date + time both extracted).
    """
    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return dt.dt.tz_convert(None)


def parse_ts_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'ts' column. Handles all observed formats:

      Already parsed by pandas (most common when reading Excel):
        Timestamp('2026-03-12 09:58:48') or datetime.datetime(2026, 3, 14, 19, 8, 27)
        → .time() is called directly, no string parsing needed.

      String fallback:
        "10-03-26 13:30"  DD-MM-YY HH:MM
        "7:08:27 PM"      12-hour AM/PM
        "19:08:27"        24-hour time-only

    Returns a Series of datetime.time objects.
    """
    import datetime as _dt

    def _parse_one(val):
        if val is None:
            return pd.NaT
        try:
            if pd.isna(val):
                return pd.NaT
        except (TypeError, ValueError):
            pass

        # Already a datetime / Timestamp — just extract time
        if isinstance(val, (pd.Timestamp, _dt.datetime)):
            return val.time()
        if isinstance(val, _dt.time):
            return val

        # String fallback
        s = str(val).strip()
        if re.match(r"^\d{2}-\d{2}-\d{2}\s+\d{1,2}:\d{2}", s):
            try:
                return pd.to_datetime(s, format="%d-%m-%y %H:%M").time()
            except Exception:
                pass
        for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
            try:
                return pd.to_datetime(s, format=fmt).time()
            except Exception:
                continue
        return pd.NaT

    return series.apply(_parse_one)


# =============================================================================
# DATA LOADING
# =============================================================================

def load_data(filepath: str, sheets: list) -> dict:
    """
    Load & clean data from specified sheets.

    Datetime used for plotting is constructed as:
      date  from 'timestamp' column  (ISO8601 UTC string)
      time  from 'ts' column         (DD-MM-YY HH:MM  OR  H:MM:SS AM/PM)

    The combined datetime is stored in column 'plot_dt'.
    """
    dfs = {}
    for sheet in sheets:
        df = pd.read_excel(filepath, sheet_name=sheet)
        df.columns = df.columns.str.strip().str.lower()

        # ── Extract date from 'timestamp' ────────────────────────────────────
        if "timestamp" not in df.columns:
            raise KeyError(f"Sheet '{sheet}' has no 'timestamp' column. "
                           f"Found: {df.columns.tolist()}")
        full_dt = parse_timestamp_col(df["timestamp"])
        date_part = full_dt.dt.date   # datetime.date objects

        # ── Extract time from 'ts' ────────────────────────────────────────────
        if "ts" not in df.columns:
            raise KeyError(f"Sheet '{sheet}' has no 'ts' column. "
                           f"Found: {df.columns.tolist()}")
        time_part = parse_ts_col(df["ts"])   # datetime.time objects

        # ── Combine date + time into a single plot_dt column ──────────────────
        def _combine(row):
            d, t = row["_date"], row["_time"]
            if pd.isna(d) or pd.isna(t) or t is pd.NaT:
                return pd.NaT
            try:
                return pd.Timestamp.combine(d, t)
            except Exception:
                return pd.NaT

        df["_date"] = date_part
        df["_time"] = time_part
        df["plot_dt"] = df.apply(_combine, axis=1)
        df = df.drop(columns=["_date", "_time"])

        df = df.sort_values("plot_dt").reset_index(drop=True)

        # Keep only the columns we need
        poll_cols = [v for v in POLLUTANTS.values() if v in df.columns]
        df = df[["plot_dt"] + poll_cols]
        dfs[sheet] = df

    return dfs


# =============================================================================
# HELPERS
# =============================================================================

def apply_style(ax, xlabel="Date & Time", ylabel="Concentration (µg/m³)"):
    s = STYLE
    ax.set_xlabel(xlabel, fontsize=s["xlabel_size"],
                  fontweight=s["xlabel_weight"], labelpad=8)
    ax.set_ylabel(ylabel, fontsize=s["ylabel_size"],
                  fontweight=s["ylabel_weight"], labelpad=8)
    ax.tick_params(axis="both", which="major",
                   labelsize=s["xtick_labelsize"],
                   length=s["tick_major_len"], width=s["tick_width"])
    ax.tick_params(axis="y", which="major",
                   labelsize=s["ytick_labelsize"])
    ax.tick_params(axis="both", which="minor",
                   length=s["tick_minor_len"], width=s["tick_width"])
    ax.grid(True, which="major", linestyle="--",
            alpha=s["grid_alpha"], linewidth=s["grid_lw"])
    ax.grid(True, which="minor", linestyle=":",
            alpha=s["grid_alpha"] * 0.6, linewidth=s["grid_lw"] * 0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(s["spine_lw"])


def apply_log_y(ax):
    """Switch Y axis to log scale with sensible ticks."""
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10, numticks=10))
    ax.yaxis.set_major_formatter(LogFormatter(base=10, labelOnlyBase=False))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs="auto", numticks=10))


def set_linear_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin_linear"], ymax)
    ax.yaxis.set_minor_locator(AutoMinorLocator())


def set_log_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin_log"], ymax)


def format_xaxis_datetime(ax, locator, formatter):
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    plt.setp(ax.get_xticklabels(),
             rotation=STYLE["xtick_rotation"], ha="right",
             fontsize=STYLE["xtick_labelsize"])


def add_legend(ax):
    s = STYLE
    ax.legend(
        title=s["legend_title"],
        fontsize=s["legend_fontsize"],
        title_fontsize=s["legend_title_size"],
        framealpha=s["legend_framealpha"],
        edgecolor=s["legend_edgecolor"],
        loc=s["legend_loc"],
    )


def save_fig(fig, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=STYLE["dpi"], bbox_inches="tight",
                format="jpeg",
                pil_kwargs={"quality": STYLE["jpg_quality"]})
    plt.close(fig)
    print(f"  Saved → {os.path.basename(path)}")


def make_figure():
    return plt.subplots(figsize=(STYLE["fig_width"], STYLE["fig_height"]))


# =============================================================================
# PLOT FUNCTIONS
# =============================================================================

# ── A) Full raw time series (linear or log) ───────────────────────────────────

def plot_full_timeseries(dfs: dict, poll_col: str, out_path: str,
                         log_scale: bool = False):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        ax.plot(df["plot_dt"], df[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    # X axis: data spans 20-25 days → tick every 2 days
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)

    # Pick ymax based on pollutant column name
    is_pm25 = (poll_col == POLLUTANTS.get("PM2.5", "pm2.5"))
    if log_scale:
        apply_log_y(ax)
        ymax = STYLE["ymax_log_full_raw_pm25"] if is_pm25 else STYLE["ymax_log_full_raw_pm10"]
        set_log_ylim(ax, ymax)
    else:
        ymax = STYLE["ymax_full_raw_pm25"] if is_pm25 else STYLE["ymax_full_raw_pm10"]
        set_linear_ylim(ax, ymax)

    fig.tight_layout()
    save_fig(fig, out_path)


# ── B) Per-day time series (linear or log) ────────────────────────────────────

def plot_daily_timeseries(dfs: dict, poll_col: str, day: pd.Timestamp,
                          out_path: str, log_scale: bool = False):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        mask = df["plot_dt"].dt.date == day.date()
        day_df = df[mask]
        if day_df.empty:
            continue
        ax.plot(day_df["plot_dt"], day_df[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax, xlabel=f"Time on {day.strftime('%d %b %Y')}")
    format_xaxis_datetime(
        ax,
        mdates.HourLocator(interval=2),
        mdates.DateFormatter("%H:%M"),
    )
    add_legend(ax)

    # Pick ymax based on pollutant column name
    is_pm25 = (poll_col == POLLUTANTS.get("PM2.5", "pm2.5"))
    if log_scale:
        apply_log_y(ax)
        ymax = STYLE["ymax_log_per_day_pm25"] if is_pm25 else STYLE["ymax_log_per_day_pm10"]
        set_log_ylim(ax, ymax)
    else:
        ymax = STYLE["ymax_per_day_pm25"] if is_pm25 else STYLE["ymax_per_day_pm10"]
        set_linear_ylim(ax, ymax)

    fig.tight_layout()
    save_fig(fig, out_path)


# ── C & D) Resampled time series (hourly / half-hourly, linear only) ──────────

def plot_resampled_timeseries(dfs: dict, poll_col: str,
                              resample_rule: str, out_path: str,
                              ymax: int):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        if poll_col not in df.columns:
            continue
        resampled = (
            df.set_index("plot_dt")[poll_col]
              .resample(resample_rule)
              .mean()
              .dropna()
              .reset_index()
        )
        ax.plot(resampled["plot_dt"], resampled[poll_col],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)
    set_linear_ylim(ax, ymax)
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    fig.tight_layout()
    save_fig(fig, out_path)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading data …")
    s_dfs = load_data(INPUT_FILE, S_SHEETS)
    c_dfs = load_data(INPUT_FILE, C_SHEETS)

    series_map = {"S": s_dfs, "C": c_dfs}

    # ── Diagnostics: parse summary + unparsed row report ────────────────────
    print("\n── PARSE DIAGNOSTICS ──────────────────────────────────────────────────")
    for series_tag, dfs in series_map.items():
        for sheet, df in dfs.items():
            raw = pd.read_excel(INPUT_FILE, sheet_name=sheet)
            raw.columns = raw.columns.str.strip().str.lower()

            ts_raw   = raw["timestamp"].iloc[0] if "timestamp" in raw.columns else "MISSING COLUMN"
            tscol    = raw["ts"].iloc[0]         if "ts"        in raw.columns else "MISSING COLUMN"
            n_valid  = df["plot_dt"].dropna().shape[0]
            n_total  = df.shape[0]
            n_failed = n_total - n_valid
            sample   = df["plot_dt"].dropna().iloc[0] if n_valid > 0 else "ALL NaT — parse failed"

            print(f"  [{sheet}]")
            print(f"    timestamp[0] raw  : {repr(ts_raw)}")
            print(f"    ts[0]        raw  : {repr(tscol)}")
            print(f"    plot_dt[0] parsed : {sample}")
            print(f"    valid rows        : {n_valid} / {n_total}  |  failed: {n_failed}")

            if n_valid == 0:
                print(f"    *** WARNING: 0 rows parsed — check format above ***")

            # ── Report unparsed rows ──────────────────────────────────────────
            if n_failed > 0:
                # Identify which rows failed: plot_dt is NaT
                failed_mask = df["plot_dt"].isna()
                # Pull original raw ts values for those rows
                raw_ts_failed  = raw.loc[failed_mask, "ts"]   if "ts"        in raw.columns else pd.Series(dtype=object)
                raw_stamp_fail = raw.loc[failed_mask, "timestamp"] if "timestamp" in raw.columns else pd.Series(dtype=object)

                # Show unique failed ts types/values (capped at 10 to keep output clean)
                unique_failed = raw_ts_failed.apply(lambda v: f"{type(v).__name__}: {repr(v)}").unique()
                print(f"    Unparsed rows ({n_failed} total) — unique raw 'ts' values seen:")
                for uv in unique_failed[:10]:
                    count = (raw_ts_failed.apply(lambda v: f"{type(v).__name__}: {repr(v)}") == uv).sum()
                    print(f"      {count:6d} rows  →  {uv}")
                if len(unique_failed) > 10:
                    print(f"      ... and {len(unique_failed) - 10} more unique value types (showing first 10 only)")

                # Show first 5 full row details (row index, timestamp, ts, pollutant values)
                failed_idx = df.index[failed_mask][:5]
                print(f"    First 5 unparsed rows (original Excel row numbers):")
                poll_cols = [v for v in POLLUTANTS.values() if v in raw.columns]
                for idx in failed_idx:
                    ts_val  = raw.loc[idx, "timestamp"] if "timestamp" in raw.columns else "—"
                    ts_col  = raw.loc[idx, "ts"]         if "ts"        in raw.columns else "—"
                    polls   = {c: raw.loc[idx, c] for c in poll_cols}
                    print(f"      Row {idx+2:6d}  |  timestamp: {ts_val}  |  ts: {repr(ts_col)}  |  {polls}")

    print("───────────────────────────────────────────────────────────────────────\n")

    # Derive date range from the loaded data itself
    all_dates_set = set()
    for dfs in series_map.values():
        for df in dfs.values():
            all_dates_set.update(df["plot_dt"].dropna().dt.date.unique())
    all_dates = sorted(all_dates_set)

    if not all_dates:
        print("\n❌  ERROR: No valid dates found across any sheet.")
        print("    Check the PARSE DIAGNOSTICS output above.")
        print("    Common causes:")
        print("      • Column not named exactly 'timestamp' or 'ts' in your Excel")
        print("      • 'timestamp' format is not ISO8601 e.g. '2026-03-10T08:02:48.202Z'")
        print("      • 'ts' format is not '10-03-26 13:30' or '7:08:27 PM'")
        print("      • Paste the actual first-row values shown above and share them")
        return

    print(f"  Date range in data: {all_dates[0]} → {all_dates[-1]} "
          f"({len(all_dates)} days)")

    resample_map = {
        "hourly":     ("1h",    STYLE["ymax_hourly"]),
        "halfhourly": ("30min", STYLE["ymax_halfhourly"]),
    }

    total = 0

    for poll_label, poll_col in POLLUTANTS.items():
        poll_tag = poll_label.replace(".", "")  # PM25 / PM10

        for series_tag, dfs in series_map.items():
            base = os.path.join(OUTPUT_DIR, series_tag + "_series")

            # ── A) Full raw — linear ──────────────────────────────────────────
            fname = f"{series_tag}_{poll_tag}_full_raw.jpg"
            out   = os.path.join(base, "full_raw", fname)
            print(f"\n[A-linear] Full raw | {series_tag} | {poll_label}")
            plot_full_timeseries(dfs, poll_col, out, log_scale=False)
            total += 1

            # ── A) Full raw — log ─────────────────────────────────────────────
            fname_log = f"{series_tag}_{poll_tag}_full_raw_log.jpg"
            out_log   = os.path.join(base, "full_raw_log", fname_log)
            print(f"[A-log]    Full raw log | {series_tag} | {poll_label}")
            plot_full_timeseries(dfs, poll_col, out_log, log_scale=True)
            total += 1

            # ── B) Per-day — linear & log ─────────────────────────────────────
            for day in all_dates:
                day_ts  = pd.Timestamp(day)
                day_str = day_ts.strftime("%Y-%m-%d")

                # linear
                fname = f"{series_tag}_{poll_tag}_{day_str}.jpg"
                out   = os.path.join(base, "per_day", fname)
                print(f"[B-linear] Per-day | {series_tag} | {poll_label} | {day_str}")
                plot_daily_timeseries(dfs, poll_col, day_ts, out, log_scale=False)
                total += 1

                # log
                fname_log = f"{series_tag}_{poll_tag}_{day_str}_log.jpg"
                out_log   = os.path.join(base, "per_day_log", fname_log)
                print(f"[B-log]    Per-day log | {series_tag} | {poll_label} | {day_str}")
                plot_daily_timeseries(dfs, poll_col, day_ts, out_log, log_scale=True)
                total += 1

            # ── C & D) Resampled ──────────────────────────────────────────────
            for res_label, (res_rule, ymax) in resample_map.items():
                fname = f"{series_tag}_{poll_tag}_{res_label}.jpg"
                out   = os.path.join(base, res_label, fname)
                print(f"[C/D] {res_label:12s} | {series_tag} | {poll_label}")
                plot_resampled_timeseries(dfs, poll_col, res_rule, out, ymax)
                total += 1

    print(f"\n✅  Done — {total} plots saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()
    

Loading data …

── PARSE DIAGNOSTICS ──────────────────────────────────────────────────
  [S1_Dharavi]
    timestamp[0] raw  : '2026-03-12T04:30:47.046Z'
    ts[0]        raw  : Timestamp('2026-03-12 09:58:48')
    plot_dt[0] parsed : 2026-03-12 00:00:28
    valid rows        : 47095 / 47095  |  failed: 0
  [S2_AntopHill]
    timestamp[0] raw  : '2026-03-05T07:15:13.701Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:41:43')
    plot_dt[0] parsed : 2026-03-05 00:00:26
    valid rows        : 65116 / 65116  |  failed: 0
  [S3_CAAQMS]
    timestamp[0] raw  : '2026-03-05T07:15:09.425Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:43:23')
    plot_dt[0] parsed : 2026-03-05 00:00:13
    valid rows        : 53716 / 53716  |  failed: 0
  [S4_Chembur]
    timestamp[0] raw  : '2026-03-05T07:14:56.479Z'
    ts[0]        raw  : Timestamp('2026-01-29 07:54:19')
    plot_dt[0] parsed : 2026-03-05 00:00:16
    valid rows        : 71307 / 71307  |  failed: 0
  [S5_KurlaEast]
    timestamp[0] r

#### 2. Ratio_Plots

In [21]:
"""
Air Quality PM2.5/PM10 Ratio Time Series Plotting Script
=========================================================
Generates ratio plots (PM2.5 ÷ PM10) from an Excel file with 9 sheets:
  S-series: S1_Dharavi, S2_AntopHill, S3_CAAQMS, S4_Chembur
  C-series: C1_Marol, C2_KurlaWest, C3_KhairaniRd, C4_Ghatkopar, C5_Powai

Plot categories (ratio only — no individual pollutant plots):
  A) Full raw time series          (2 series)
  B) Per-day time series           (days × 2 series)
  C) Hourly averaged time series   (2 series)
  D) Half-hourly averaged t-series (2 series)

Output folder structure:
  OUTPUT_DIR/
    S_series/
      full_raw/
      per_day/
      hourly/
      halfhourly/
    C_series/
      (same sub-folders)

Date source : 'timestamp' column  e.g. "2026-03-10T08:02:48.202Z"
Time source : 'ts' column         e.g. Timestamp / datetime object  OR
                                        "10-03-26 13:30" / "7:08:27 PM" strings

Ratio notes:
  - Computed as PM2.5 / PM10 row-by-row BEFORE any resampling
  - Rows where PM10 == 0 or is NaN are excluded (division undefined)
  - Ratio is expected in range 0–1 for typical ambient conditions
    (fine fraction cannot exceed coarse; values > 1 may indicate sensor issues)
"""

import os
import re
import datetime as _dt
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

# =============================================================================
# USER CONFIGURATION — edit paths here
# =============================================================================
INPUT_FILE = r"D:/Hyperlocal_Study/Data_Analysis/Sample_Data_Mar/30_Mar_Data_LAMP.xlsx"
OUTPUT_DIR = r"C:/Users/Atique/LAMP_Coding/Visualisation/Data_Quality_Assessment/Mar30/Ratio_Plots/"
# =============================================================================

# ── Sheet groups ──────────────────────────────────────────────────────────────
S_SHEETS = ["S1_Dharavi", "S2_AntopHill", "S3_CAAQMS", "S4_Chembur","S5_KurlaEast"]
C_SHEETS = ["C1_Marol", "C2_KurlaWest", "C3_KhairaniRd", "C4_Ghatkopar", "C5_Powai"]

# Raw column names for the two pollutants used in the ratio
PM25_COL = "pm2.5"
PM10_COL = "pm10"

# ── Colour palette (one colour per sheet, up to 9) ───────────────────────────
COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22",
]

# =============================================================================
# STYLE CONFIGURATION — customise everything here
# =============================================================================
STYLE = dict(

    # ── Figure ────────────────────────────────────────────────────────────────
    fig_width           = 16,
    fig_height          = 5,
    dpi                 = 150,
    jpg_quality         = 95,

    # ── Lines ─────────────────────────────────────────────────────────────────
    linewidth           = 1.2,

    # ── Axis labels ───────────────────────────────────────────────────────────
    xlabel_size         = 13,
    xlabel_weight       = "normal",       # "normal" or "bold"
    ylabel_size         = 13,
    ylabel_weight       = "normal",

    # ── Tick labels ───────────────────────────────────────────────────────────
    xtick_labelsize     = 12,
    ytick_labelsize     = 12,
    xtick_rotation      = 30,

    # ── Tick marks ────────────────────────────────────────────────────────────
    tick_major_len      = 6,
    tick_minor_len      = 3,
    tick_width          = 1.0,

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_fontsize     = 12,
    legend_title_size   = 12,
    legend_title        = "Station",
    legend_framealpha   = 0.85,
    legend_edgecolor    = "grey",
    legend_loc          = "upper right",

    # ── Grid ──────────────────────────────────────────────────────────────────
    grid_alpha          = 0.3,
    grid_lw             = 0.7,

    # ── Spines ────────────────────────────────────────────────────────────────
    spine_lw            = 1.1,

    # ── Y-axis fixed limits for ratio plots ───────────────────────────────────
    # Ratio = PM2.5 / PM10.  Physically constrained to 0–1 for ambient data.
    # Raise ymax_ratio above 1.0 only if your sensor occasionally reads > 1.
    ymin_ratio          = 0.0,    # bottom of Y axis (all plot types)
    ymax_full_raw       = 1.5,    # full raw ratio plot
    ymax_per_day        = 1.5,    # per-day ratio plot
    ymax_hourly         = 1.2,    # hourly averaged ratio plot
    ymax_halfhourly     = 1.2,    # half-hourly averaged ratio plot
)

YLABEL = "PM2.5 / PM10 ratio"


# =============================================================================
# TIMESTAMP PARSING  (identical to main plotting script)
# =============================================================================

def parse_timestamp_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'timestamp' column (ISO8601 UTC string).
    Returns tz-naive datetime Series — date component is used for plotting.
    """
    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return dt.dt.tz_convert(None)


def parse_ts_col(series: pd.Series) -> pd.Series:
    """
    Parse the 'ts' column for the TIME component.

    Handles:
      • pandas Timestamp  e.g. Timestamp('2026-03-12 09:58:48')
      • datetime.datetime e.g. datetime.datetime(2026, 3, 14, 19, 8, 27)
      • string "10-03-26 13:30"   DD-MM-YY HH:MM
      • string "7:08:27 PM"       12-hour AM/PM
      • string "19:08:27"         24-hour time-only

    Returns a Series of datetime.time objects.
    """
    def _parse_one(val):
        if val is None:
            return pd.NaT
        try:
            if pd.isna(val):
                return pd.NaT
        except (TypeError, ValueError):
            pass

        # Already a datetime / Timestamp — extract .time() directly
        if isinstance(val, (pd.Timestamp, _dt.datetime)):
            return val.time()
        if isinstance(val, _dt.time):
            return val

        # String fallback
        s = str(val).strip()
        if re.match(r"^\d{2}-\d{2}-\d{2}\s+\d{1,2}:\d{2}", s):
            try:
                return pd.to_datetime(s, format="%d-%m-%y %H:%M").time()
            except Exception:
                pass
        for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
            try:
                return pd.to_datetime(s, format=fmt).time()
            except Exception:
                continue
        return pd.NaT

    return series.apply(_parse_one)


# =============================================================================
# DATA LOADING
# =============================================================================

def load_data(filepath: str, sheets: list) -> dict:
    """
    Load each sheet, parse timestamps, compute PM2.5/PM10 ratio.

    Returns a dict  { sheet_name: DataFrame }
    Each DataFrame has columns:
        plot_dt   — combined date (from 'timestamp') + time (from 'ts')
        ratio     — PM2.5 / PM10  (NaN where PM10 is 0 or missing)
    """
    dfs = {}
    for sheet in sheets:
        df = pd.read_excel(filepath, sheet_name=sheet)
        df.columns = df.columns.str.strip().str.lower()

        # ── Date from 'timestamp' ────────────────────────────────────────────
        if "timestamp" not in df.columns:
            raise KeyError(f"[{sheet}] No 'timestamp' column. Found: {df.columns.tolist()}")
        full_dt   = parse_timestamp_col(df["timestamp"])
        date_part = full_dt.dt.date

        # ── Time from 'ts' ───────────────────────────────────────────────────
        if "ts" not in df.columns:
            raise KeyError(f"[{sheet}] No 'ts' column. Found: {df.columns.tolist()}")
        time_part = parse_ts_col(df["ts"])

        # ── Combine → plot_dt ────────────────────────────────────────────────
        def _combine(row):
            d, t = row["_date"], row["_time"]
            if pd.isna(d) or pd.isna(t) or t is pd.NaT:
                return pd.NaT
            try:
                return pd.Timestamp.combine(d, t)
            except Exception:
                return pd.NaT

        df["_date"]   = date_part
        df["_time"]   = time_part
        df["plot_dt"] = df.apply(_combine, axis=1)
        df = df.drop(columns=["_date", "_time"])

        # ── Compute ratio ────────────────────────────────────────────────────
        if PM25_COL not in df.columns or PM10_COL not in df.columns:
            raise KeyError(
                f"[{sheet}] Missing pollutant columns. "
                f"Need '{PM25_COL}' and '{PM10_COL}'. "
                f"Found: {df.columns.tolist()}"
            )
        pm25 = pd.to_numeric(df[PM25_COL], errors="coerce")
        pm10 = pd.to_numeric(df[PM10_COL], errors="coerce")

        # Exclude rows where PM10 is zero or NaN (ratio undefined)
        ratio = pm25 / pm10.replace(0, float("nan"))

        df["ratio"] = ratio

        df = df[["plot_dt", "ratio"]].sort_values("plot_dt").reset_index(drop=True)
        dfs[sheet] = df

    return dfs


# =============================================================================
# STYLE HELPERS
# =============================================================================

def apply_style(ax, xlabel="Date & Time"):
    s = STYLE
    ax.set_xlabel(xlabel, fontsize=s["xlabel_size"],
                  fontweight=s["xlabel_weight"], labelpad=8)
    ax.set_ylabel(YLABEL, fontsize=s["ylabel_size"],
                  fontweight=s["ylabel_weight"], labelpad=8)
    ax.tick_params(axis="both", which="major",
                   labelsize=s["xtick_labelsize"],
                   length=s["tick_major_len"], width=s["tick_width"])
    ax.tick_params(axis="y", which="major",
                   labelsize=s["ytick_labelsize"])
    ax.tick_params(axis="both", which="minor",
                   length=s["tick_minor_len"], width=s["tick_width"])
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, which="major", linestyle="--",
            alpha=s["grid_alpha"], linewidth=s["grid_lw"])
    ax.grid(True, which="minor", linestyle=":",
            alpha=s["grid_alpha"] * 0.6, linewidth=s["grid_lw"] * 0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(s["spine_lw"])


def format_xaxis_datetime(ax, locator, formatter):
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    plt.setp(ax.get_xticklabels(),
             rotation=STYLE["xtick_rotation"], ha="right",
             fontsize=STYLE["xtick_labelsize"])


def add_legend(ax):
    s = STYLE
    ax.legend(
        title=s["legend_title"],
        fontsize=s["legend_fontsize"],
        title_fontsize=s["legend_title_size"],
        framealpha=s["legend_framealpha"],
        edgecolor=s["legend_edgecolor"],
        loc=s["legend_loc"],
    )


def set_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin_ratio"], ymax)


def save_fig(fig, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=STYLE["dpi"], bbox_inches="tight",
                format="jpeg",
                pil_kwargs={"quality": STYLE["jpg_quality"]})
    plt.close(fig)
    print(f"  Saved → {os.path.basename(path)}")


def make_figure():
    return plt.subplots(figsize=(STYLE["fig_width"], STYLE["fig_height"]))


# =============================================================================
# PLOT FUNCTIONS
# =============================================================================

# ── A) Full raw ratio time series ─────────────────────────────────────────────

def plot_full_raw(dfs: dict, out_path: str):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        valid = df.dropna(subset=["ratio"])
        if valid.empty:
            continue
        ax.plot(valid["plot_dt"], valid["ratio"],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)
    set_ylim(ax, STYLE["ymax_full_raw"])
    fig.tight_layout()
    save_fig(fig, out_path)


# ── B) Per-day ratio time series ──────────────────────────────────────────────

def plot_per_day(dfs: dict, day: pd.Timestamp, out_path: str):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        mask   = df["plot_dt"].dt.date == day.date()
        day_df = df[mask].dropna(subset=["ratio"])
        if day_df.empty:
            continue
        ax.plot(day_df["plot_dt"], day_df["ratio"],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax, xlabel=f"Time on {day.strftime('%d %b %Y')}")
    format_xaxis_datetime(
        ax,
        mdates.HourLocator(interval=2),
        mdates.DateFormatter("%H:%M"),
    )
    add_legend(ax)
    set_ylim(ax, STYLE["ymax_per_day"])
    fig.tight_layout()
    save_fig(fig, out_path)


# ── C & D) Resampled ratio (hourly / half-hourly) ─────────────────────────────

def plot_resampled(dfs: dict, resample_rule: str, out_path: str, ymax: float):
    fig, ax = make_figure()
    for i, (sheet, df) in enumerate(dfs.items()):
        resampled = (
            df.set_index("plot_dt")["ratio"]
              .resample(resample_rule)
              .mean()
              .dropna()
              .reset_index()
        )
        if resampled.empty:
            continue
        ax.plot(resampled["plot_dt"], resampled["ratio"],
                label=sheet, color=COLORS[i % len(COLORS)],
                linewidth=STYLE["linewidth"])

    apply_style(ax)
    format_xaxis_datetime(
        ax,
        mdates.DayLocator(interval=2),
        mdates.DateFormatter("%d %b\n%Y"),
    )
    add_legend(ax)
    set_ylim(ax, ymax)
    fig.tight_layout()
    save_fig(fig, out_path)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading data …")
    s_dfs = load_data(INPUT_FILE, S_SHEETS)
    c_dfs = load_data(INPUT_FILE, C_SHEETS)

    series_map = {"S": s_dfs, "C": c_dfs}

    # ── Diagnostics ──────────────────────────────────────────────────────────
    print("\n── PARSE DIAGNOSTICS ──────────────────────────────────────────────────")
    for series_tag, dfs in series_map.items():
        for sheet, df in dfs.items():
            raw = pd.read_excel(INPUT_FILE, sheet_name=sheet)
            raw.columns = raw.columns.str.strip().str.lower()

            ts_raw  = raw["timestamp"].iloc[0] if "timestamp" in raw.columns else "MISSING"
            ts_col  = raw["ts"].iloc[0]         if "ts"        in raw.columns else "MISSING"
            n_total = df.shape[0]
            n_valid = df["plot_dt"].dropna().shape[0]
            n_ratio = df["ratio"].dropna().shape[0]
            n_failed = n_total - n_valid

            sample  = df["plot_dt"].dropna().iloc[0] if n_valid > 0 else "ALL NaT — parse failed"

            print(f"  [{sheet}]")
            print(f"    timestamp[0] raw  : {repr(ts_raw)}")
            print(f"    ts[0]        raw  : {repr(ts_col)}")
            print(f"    plot_dt[0] parsed : {sample}")
            print(f"    valid rows        : {n_valid} / {n_total}  |  failed: {n_failed}")
            print(f"    ratio rows valid  : {n_ratio} / {n_total}  "
                  f"(excludes PM10=0 or NaN in addition to parse failures)")

            if n_valid == 0:
                print(f"    *** WARNING: 0 rows parsed — check timestamp/ts formats ***")

            # Report unparsed rows if any
            if n_failed > 0:
                failed_mask   = df["plot_dt"].isna()
                raw_ts_failed = raw.loc[failed_mask, "ts"] if "ts" in raw.columns else pd.Series(dtype=object)
                unique_failed = raw_ts_failed.apply(
                    lambda v: f"{type(v).__name__}: {repr(v)}"
                ).unique()
                print(f"    Unparsed rows ({n_failed}) — unique raw 'ts' values:")
                for uv in unique_failed[:10]:
                    count = (raw_ts_failed.apply(
                        lambda v: f"{type(v).__name__}: {repr(v)}") == uv).sum()
                    print(f"      {count:6d} rows  →  {uv}")
                if len(unique_failed) > 10:
                    print(f"      ... and {len(unique_failed)-10} more (showing first 10)")

                failed_idx = df.index[failed_mask][:5]
                poll_cols  = [c for c in [PM25_COL, PM10_COL] if c in raw.columns]
                print(f"    First 5 unparsed rows:")
                for idx in failed_idx:
                    ts_v = raw.loc[idx, "timestamp"] if "timestamp" in raw.columns else "—"
                    ts_c = raw.loc[idx, "ts"]         if "ts"        in raw.columns else "—"
                    polls = {c: raw.loc[idx, c] for c in poll_cols}
                    print(f"      Row {idx+2:6d}  |  timestamp: {ts_v}  |  ts: {repr(ts_c)}  |  {polls}")

    print("───────────────────────────────────────────────────────────────────────\n")

    # ── Date range ───────────────────────────────────────────────────────────
    all_dates_set = set()
    for dfs in series_map.values():
        for df in dfs.values():
            all_dates_set.update(df["plot_dt"].dropna().dt.date.unique())
    all_dates = sorted(all_dates_set)

    if not all_dates:
        print("❌  ERROR: No valid dates found. Check diagnostics above.")
        return

    print(f"  Date range: {all_dates[0]} → {all_dates[-1]} ({len(all_dates)} days)\n")

    resample_map = {
        "hourly":     ("1h",    STYLE["ymax_hourly"]),
        "halfhourly": ("30min", STYLE["ymax_halfhourly"]),
    }

    total = 0

    for series_tag, dfs in series_map.items():
        base = os.path.join(OUTPUT_DIR, series_tag + "_series")

        # ── A) Full raw ───────────────────────────────────────────────────────
        out = os.path.join(base, "full_raw", f"{series_tag}_ratio_full_raw.jpg")
        print(f"[A] Full raw | {series_tag}")
        plot_full_raw(dfs, out)
        total += 1

        # ── B) Per-day ────────────────────────────────────────────────────────
        for day in all_dates:
            day_ts  = pd.Timestamp(day)
            day_str = day_ts.strftime("%Y-%m-%d")
            out = os.path.join(base, "per_day", f"{series_tag}_ratio_{day_str}.jpg")
            print(f"[B] Per-day  | {series_tag} | {day_str}")
            plot_per_day(dfs, day_ts, out)
            total += 1

        # ── C & D) Resampled ──────────────────────────────────────────────────
        for res_label, (res_rule, ymax) in resample_map.items():
            out = os.path.join(base, res_label, f"{series_tag}_ratio_{res_label}.jpg")
            print(f"[C/D] {res_label:12s} | {series_tag}")
            plot_resampled(dfs, res_rule, out, ymax)
            total += 1

    print(f"\n✅  Done — {total} plots saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Loading data …

── PARSE DIAGNOSTICS ──────────────────────────────────────────────────
  [S1_Dharavi]
    timestamp[0] raw  : '2026-03-12T04:30:47.046Z'
    ts[0]        raw  : Timestamp('2026-03-12 09:58:48')
    plot_dt[0] parsed : 2026-03-12 00:00:28
    valid rows        : 47095 / 47095  |  failed: 0
    ratio rows valid  : 47095 / 47095  (excludes PM10=0 or NaN in addition to parse failures)
  [S2_AntopHill]
    timestamp[0] raw  : '2026-03-05T07:15:13.701Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:41:43')
    plot_dt[0] parsed : 2026-03-05 00:00:26
    valid rows        : 65116 / 65116  |  failed: 0
    ratio rows valid  : 65116 / 65116  (excludes PM10=0 or NaN in addition to parse failures)
  [S3_CAAQMS]
    timestamp[0] raw  : '2026-03-05T07:15:09.425Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:43:23')
    plot_dt[0] parsed : 2026-03-05 00:00:13
    valid rows        : 53716 / 53716  |  failed: 0
    ratio rows valid  : 53716 / 53716  (excludes PM10=0 or NaN in 

#### 3. Acc to Days type

In [24]:
"""
Air Quality Day-Type Comparison Time Series Script
====================================================
Plots PM2.5 and PM10 as continuous time series, colour-coded by station
and marker-coded by day type:

  Day types:
    Weekday  (Mon/Tue/Wed/Thu) — averaged across all matching days
    Friday
    Saturday
    Sunday

  For each day type:
    • The trace is the mean concentration at each time-of-day slot
      (averaged across all days of that type in the study period)
    • X-axis = time of day (00:00 → 24:00)
    • Each station gets its own colour; day type shown via marker style

  Legend is split into two parts displayed together:
    • Colour patches  → station names
    • Marker symbols  → day type labels

  Outputs (no per-day plots):
    OUTPUT_DIR/
      S_series/
        full_raw/        ← raw resolution averaged by day type
        hourly/
        halfhourly/
      C_series/
        (same)

  Y-axis limits (uniform, irrespective of data):
    PM10  full_raw  → 1800    PM2.5  full_raw  → 1000
    PM10  hourly    →  400    PM2.5  hourly    →  400
    PM10  halfhourly→  400    PM2.5  halfhourly→  400

Date source : 'timestamp' column  e.g. "2026-03-10T08:02:48.202Z"
Time source : 'ts' column         pandas Timestamp / datetime / string
"""

import os
import re
import datetime as _dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

# =============================================================================
# USER CONFIGURATION — edit paths here
# =============================================================================
INPUT_FILE = r"D:/Hyperlocal_Study/Data_Analysis/Sample_Data_Mar/30_Mar_Data_LAMP.xlsx"
OUTPUT_DIR = r"C:/Users/Atique/LAMP_Coding/Visualisation/Data_Quality_Assessment/Mar30/Day_Types/"
# =============================================================================

# ── Sheet groups ──────────────────────────────────────────────────────────────
S_SHEETS = ["S1_Dharavi", "S2_AntopHill", "S3_CAAQMS", "S4_Chembur","S5_KurlaEast"]
C_SHEETS = ["C1_Marol", "C2_KurlaWest", "C3_KhairaniRd", "C4_Ghatkopar", "C5_Powai"]

POLLUTANTS = {
    "PM2.5": "pm2.5",
    "PM10":  "pm10",
}

# =============================================================================
# DAY-TYPE MAPPING
# Edit this dict: keys are date strings "YYYY-MM-DD", values are one of:
#   "Weekday"  "Friday"  "Saturday"  "Sunday"
# Weekday = Mon/Tue/Wed/Thu averaged together.
# =============================================================================
DAY_TYPE_MAP = {
    "2026-03-05": "Thursday",   # ← change "Thursday" to "Weekday" to include
    "2026-03-06": "Friday",
    "2026-03-07": "Saturday",
    "2026-03-08": "Sunday",
    "2026-03-09": "Weekday",    # Mon
    "2026-03-10": "Weekday",    # Tue
    "2026-03-11": "Weekday",    # Wed
    "2026-03-12": "Thursday",
    "2026-03-13": "Friday",
    "2026-03-14": "Saturday",
    "2026-03-15": "Sunday",
    "2026-03-16": "Weekday",    # Mon
    "2026-03-17": "Weekday",    # Tue
    "2026-03-18": "Weekday",    # Wed
    "2026-03-19": "Thursday",
    "2026-03-20": "Friday",
    "2026-03-21": "Saturday",
    "2026-03-22": "Sunday",
    "2026-03-23": "Weekday",    # Mon
    "2026-03-24": "Weekday",    # Tue
    "2026-03-25": "Weekday",    # Wed
    "2026-03-26": "Thursday",
    "2026-03-27": "Friday",
    "2026-03-28": "Saturday",
    "2026-03-29": "Sunday",
    "2026-03-30": "Weekday",    # Mon
}

# "Thursday" treated as Weekday — remap here so plot logic is clean
# If you want Thursday separate, add it as its own day type and update
# DAY_MARKERS and DAY_COLORS below.
THURSDAY_IS_WEEKDAY = True   # set False to keep Thursday separate

# ── Colour palette — one colour per station (up to 9) ────────────────────────
STATION_COLORS = [
    "#1f77b4",   # blue
    "#ff7f0e",   # orange
    "#2ca02c",   # green
    "#d62728",   # red
    "#9467bd",   # purple
    "#8c564b",   # brown
    "#e377c2",   # pink
    "#7f7f7f",   # grey
    "#bcbd22",   # olive
]

# ── Marker style per day type ─────────────────────────────────────────────────
# markevery controls how often markers appear along the line (1 = every point,
# higher = sparser). Increase to avoid clutter on dense raw data.
DAY_MARKERS = {
    "Weekday":  {"marker": "o",  "markersize": 3,  "markevery": 30},
    "Friday":   {"marker": "s",  "markersize": 3,  "markevery": 30},
    "Saturday": {"marker": "^",  "markersize": 3,  "markevery": 30},
    "Sunday":   {"marker": "D",  "markersize": 3,  "markevery": 30},
    "Thursday": {"marker": "P",  "markersize": 3,  "markevery": 30},
}

# ── Day type display order in legend ─────────────────────────────────────────
DAY_ORDER = ["Weekday", "Friday", "Saturday", "Sunday"]
if not THURSDAY_IS_WEEKDAY:
    DAY_ORDER.append("Thursday")

# =============================================================================
# STYLE CONFIGURATION — customise everything here
# =============================================================================
STYLE = dict(

    # ── Figure ────────────────────────────────────────────────────────────────
    fig_width           = 16,
    fig_height          = 5,
    dpi                 = 150,
    jpg_quality         = 95,

    # ── Lines ─────────────────────────────────────────────────────────────────
    linewidth           = 1.2,
    line_alpha          = 0.85,     # line transparency (0–1)

    # ── Axis labels ───────────────────────────────────────────────────────────
    xlabel_size         = 13,
    xlabel_weight       = "normal",
    ylabel_size         = 13,
    ylabel_weight       = "normal",

    # ── Tick labels ───────────────────────────────────────────────────────────
    xtick_labelsize     = 12,
    ytick_labelsize     = 12,
    xtick_rotation      = 30,

    # ── Tick marks ────────────────────────────────────────────────────────────
    tick_major_len      = 6,
    tick_minor_len      = 3,
    tick_width          = 1.0,

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_fontsize     = 11,
    legend_title_size   = 12,
    legend_framealpha   = 0.85,
    legend_edgecolor    = "grey",
    legend_loc          = "upper right",

    # ── Grid ──────────────────────────────────────────────────────────────────
    grid_alpha          = 0.3,
    grid_lw             = 0.7,

    # ── Spines ────────────────────────────────────────────────────────────────
    spine_lw            = 1.1,

    # ── Y-axis limits ─────────────────────────────────────────────────────────
    ymin                = 0,

    ymax_full_raw_pm25  = 1000,
    ymax_full_raw_pm10  = 1800,

    ymax_hourly_pm25    = 400,
    ymax_hourly_pm10    = 400,

    ymax_halfhourly_pm25 = 400,
    ymax_halfhourly_pm10 = 400,
)


# =============================================================================
# TIMESTAMP PARSING
# =============================================================================

def parse_timestamp_col(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return dt.dt.tz_convert(None)


def parse_ts_col(series: pd.Series) -> pd.Series:
    def _parse_one(val):
        if val is None:
            return pd.NaT
        try:
            if pd.isna(val):
                return pd.NaT
        except (TypeError, ValueError):
            pass
        if isinstance(val, (pd.Timestamp, _dt.datetime)):
            return val.time()
        if isinstance(val, _dt.time):
            return val
        s = str(val).strip()
        if re.match(r"^\d{2}-\d{2}-\d{2}\s+\d{1,2}:\d{2}", s):
            try:
                return pd.to_datetime(s, format="%d-%m-%y %H:%M").time()
            except Exception:
                pass
        for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
            try:
                return pd.to_datetime(s, format=fmt).time()
            except Exception:
                continue
        return pd.NaT

    return series.apply(_parse_one)


# =============================================================================
# DATA LOADING
# =============================================================================

def load_data(filepath: str, sheets: list) -> dict:
    """
    Returns dict { sheet: DataFrame } with columns:
        plot_dt   — combined datetime (date from timestamp, time from ts)
        pm2.5
        pm10
        date_str  — "YYYY-MM-DD"
        day_type  — "Weekday" / "Friday" / "Saturday" / "Sunday" / None
    """
    # Build normalised day map (remap Thursday → Weekday if configured)
    day_map = {}
    for d, dt in DAY_TYPE_MAP.items():
        if THURSDAY_IS_WEEKDAY and dt == "Thursday":
            day_map[d] = "Weekday"
        else:
            day_map[d] = dt

    dfs = {}
    for sheet in sheets:
        df = pd.read_excel(filepath, sheet_name=sheet)
        df.columns = df.columns.str.strip().str.lower()

        if "timestamp" not in df.columns:
            raise KeyError(f"[{sheet}] No 'timestamp' column.")
        if "ts" not in df.columns:
            raise KeyError(f"[{sheet}] No 'ts' column.")

        full_dt   = parse_timestamp_col(df["timestamp"])
        date_part = full_dt.dt.date
        time_part = parse_ts_col(df["ts"])

        def _combine(row):
            d, t = row["_date"], row["_time"]
            if pd.isna(d) or pd.isna(t) or t is pd.NaT:
                return pd.NaT
            try:
                return pd.Timestamp.combine(d, t)
            except Exception:
                return pd.NaT

        df["_date"]   = date_part
        df["_time"]   = time_part
        df["plot_dt"] = df.apply(_combine, axis=1)
        df = df.drop(columns=["_date", "_time"])

        df["date_str"] = df["plot_dt"].dt.strftime("%Y-%m-%d")
        df["day_type"] = df["date_str"].map(day_map)

        poll_cols = [v for v in POLLUTANTS.values() if v in df.columns]
        for c in poll_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        keep = ["plot_dt", "date_str", "day_type"] + poll_cols
        df = df[keep].sort_values("plot_dt").reset_index(drop=True)
        dfs[sheet] = df

    return dfs


# =============================================================================
# AGGREGATION — average each day type into a single time-of-day profile
# =============================================================================

def build_daytype_profiles(df: pd.DataFrame, poll_col: str,
                           resample_rule: str) -> dict:
    """
    For each day type, resample the data to resample_rule resolution,
    then average across all days of that type.

    Returns dict { day_type: DataFrame with columns [time_of_day, mean_val] }
    where time_of_day is a Timedelta from midnight (for x-axis plotting).
    """
    profiles = {}

    for day_type in DAY_ORDER:
        subset = df[df["day_type"] == day_type].copy()
        if subset.empty or poll_col not in subset.columns:
            continue

        subset = subset.dropna(subset=["plot_dt", poll_col])
        if subset.empty:
            continue

        # Resample each individual day, then average across days
        daily_profiles = []
        for date_str, grp in subset.groupby("date_str"):
            grp = grp.set_index("plot_dt")[poll_col]
            resampled = grp.resample(resample_rule).mean()
            # Normalise to time-of-day (seconds from midnight)
            resampled.index = (
                resampled.index.hour * 3600 +
                resampled.index.minute * 60 +
                resampled.index.second
            )
            daily_profiles.append(resampled)

        if not daily_profiles:
            continue

        combined = pd.concat(daily_profiles, axis=1)
        mean_profile = combined.mean(axis=1)

        # Convert seconds-from-midnight back to a plottable datetime
        # Use a fixed reference date so all day types share the same x-axis
        ref_date = pd.Timestamp("2000-01-01")
        mean_profile.index = pd.to_datetime(
            mean_profile.index, unit="s", origin=ref_date
        )
        profiles[day_type] = mean_profile.dropna()

    return profiles


# =============================================================================
# STYLE HELPERS
# =============================================================================

def apply_style(ax, xlabel="Time of day"):
    s = STYLE
    ax.set_xlabel(xlabel, fontsize=s["xlabel_size"],
                  fontweight=s["xlabel_weight"], labelpad=8)
    ax.tick_params(axis="both", which="major",
                   labelsize=s["xtick_labelsize"],
                   length=s["tick_major_len"], width=s["tick_width"])
    ax.tick_params(axis="y", which="major",
                   labelsize=s["ytick_labelsize"])
    ax.tick_params(axis="both", which="minor",
                   length=s["tick_minor_len"], width=s["tick_width"])
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, which="major", linestyle="--",
            alpha=s["grid_alpha"], linewidth=s["grid_lw"])
    ax.grid(True, which="minor", linestyle=":",
            alpha=s["grid_alpha"] * 0.6, linewidth=s["grid_lw"] * 0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(s["spine_lw"])

    # X-axis: 0–24 h labels
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=1))
    plt.setp(ax.get_xticklabels(),
             rotation=STYLE["xtick_rotation"], ha="right",
             fontsize=STYLE["xtick_labelsize"])


def set_ylabel(ax, poll_label):
    ax.set_ylabel(f"{poll_label} Concentration (µg/m³)",
                  fontsize=STYLE["ylabel_size"],
                  fontweight=STYLE["ylabel_weight"], labelpad=8)


def set_ylim(ax, ymax):
    ax.set_ylim(STYLE["ymin"], ymax)


def build_legend(ax, sheets, sheet_colors):
    """
    Two-section legend:
      Section 1 — coloured solid lines  → station names
      Section 2 — black marker symbols  → day types
    """
    s = STYLE

    # Station colour patches
    station_handles = [
        mlines.Line2D([], [], color=sheet_colors[i], linewidth=2,
                      label=sheet)
        for i, sheet in enumerate(sheets)
        if i < len(sheet_colors)
    ]

    # Day-type marker handles (black line, distinct marker)
    day_handles = []
    for dt in DAY_ORDER:
        mk = DAY_MARKERS.get(dt, {})
        h = mlines.Line2D(
            [], [],
            color="black",
            linewidth=1,
            marker=mk.get("marker", "o"),
            markersize=mk.get("markersize", 4) + 1,
            label=dt,
        )
        day_handles.append(h)

    # Blank separator
    blank = mlines.Line2D([], [], color="none", label="")

    all_handles = (
        [mpatches.Patch(color="none", label="— Station —")] +
        station_handles +
        [blank] +
        [mpatches.Patch(color="none", label="— Day type —")] +
        day_handles
    )

    ax.legend(
        handles=all_handles,
        fontsize=s["legend_fontsize"],
        title_fontsize=s["legend_title_size"],
        framealpha=s["legend_framealpha"],
        edgecolor=s["legend_edgecolor"],
        loc=s["legend_loc"],
        ncol=1,
    )


def save_fig(fig, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=STYLE["dpi"], bbox_inches="tight",
                format="jpeg",
                pil_kwargs={"quality": STYLE["jpg_quality"]})
    plt.close(fig)
    print(f"  Saved → {os.path.basename(path)}")


def make_figure():
    return plt.subplots(figsize=(STYLE["fig_width"], STYLE["fig_height"]))


# =============================================================================
# PLOT FUNCTION — one plot for one (series, pollutant, resample_rule)
# =============================================================================

def plot_daytype(dfs: dict, sheets: list, poll_col: str, poll_label: str,
                 resample_rule: str, ymax: float, out_path: str,
                 plot_title: str = ""):
    """
    For each sheet (station), compute day-type averaged profiles at
    resample_rule resolution, then plot all day types as separate lines
    with the station colour and day-type marker.
    """
    fig, ax = make_figure()

    sheet_colors = STATION_COLORS[:len(sheets)]

    for s_idx, sheet in enumerate(sheets):
        if sheet not in dfs:
            continue
        df = dfs[sheet]
        color = sheet_colors[s_idx]

        profiles = build_daytype_profiles(df, poll_col, resample_rule)

        for day_type, profile in profiles.items():
            if profile.empty:
                continue
            mk = DAY_MARKERS.get(day_type, {})
            ax.plot(
                profile.index,
                profile.values,
                color=color,
                linewidth=STYLE["linewidth"],
                alpha=STYLE["line_alpha"],
                marker=mk.get("marker", "o"),
                markersize=mk.get("markersize", 3),
                markevery=mk.get("markevery", 30),
                markeredgewidth=0.5,
                markeredgecolor="white",
            )

    apply_style(ax)
    set_ylabel(ax, poll_label)
    set_ylim(ax, ymax)

    if plot_title:
        ax.set_title(plot_title, fontsize=11, fontweight="bold", pad=8)

    build_legend(ax, sheets, sheet_colors)
    fig.tight_layout()
    save_fig(fig, out_path)


# =============================================================================
# DIAGNOSTICS
# =============================================================================

def run_diagnostics(series_map: dict):
    print("\n── PARSE DIAGNOSTICS ──────────────────────────────────────────────────")
    for series_tag, dfs in series_map.items():
        for sheet, df in dfs.items():
            raw = pd.read_excel(INPUT_FILE, sheet_name=sheet)
            raw.columns = raw.columns.str.strip().str.lower()

            ts_raw  = raw["timestamp"].iloc[0] if "timestamp" in raw.columns else "MISSING"
            ts_col  = raw["ts"].iloc[0]         if "ts"        in raw.columns else "MISSING"
            n_total = df.shape[0]
            n_valid = df["plot_dt"].dropna().shape[0]
            n_failed = n_total - n_valid
            sample  = (df["plot_dt"].dropna().iloc[0]
                       if n_valid > 0 else "ALL NaT — parse failed")

            # Day-type coverage
            dt_counts = df["day_type"].value_counts().to_dict()

            print(f"  [{sheet}]")
            print(f"    timestamp[0] raw  : {repr(ts_raw)}")
            print(f"    ts[0]        raw  : {repr(ts_col)}")
            print(f"    plot_dt[0] parsed : {sample}")
            print(f"    valid rows        : {n_valid} / {n_total}  |  failed: {n_failed}")
            print(f"    day-type counts   : {dt_counts}")

            if n_valid == 0:
                print(f"    *** WARNING: 0 rows parsed ***")

            if n_failed > 0:
                failed_mask   = df["plot_dt"].isna()
                raw_ts_failed = (raw.loc[failed_mask, "ts"]
                                 if "ts" in raw.columns
                                 else pd.Series(dtype=object))
                unique_failed = raw_ts_failed.apply(
                    lambda v: f"{type(v).__name__}: {repr(v)}"
                ).unique()
                print(f"    Unparsed rows ({n_failed}) — unique raw 'ts' values:")
                for uv in unique_failed[:5]:
                    cnt = (raw_ts_failed.apply(
                        lambda v: f"{type(v).__name__}: {repr(v)}") == uv).sum()
                    print(f"      {cnt:6d} rows  →  {uv}")

    print("───────────────────────────────────────────────────────────────────────\n")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading data …")
    s_dfs = load_data(INPUT_FILE, S_SHEETS)
    c_dfs = load_data(INPUT_FILE, C_SHEETS)

    series_map = {"S": (s_dfs, S_SHEETS), "C": (c_dfs, C_SHEETS)}

    # Diagnostics
    run_diagnostics({"S": s_dfs, "C": c_dfs})

    # Resample rules for plot types
    plot_types = {
        # key: (resample_rule, subfolder, title_suffix)
        "full_raw":   ("1min",  "full_raw",   "Raw (1-min avg)"),
        "hourly":     ("1h",    "hourly",      "Hourly avg"),
        "halfhourly": ("30min", "halfhourly",  "Half-hourly avg"),
    }

    # Y-axis limit lookup: (plot_type, poll_col) → ymax
    def get_ymax(plot_type: str, poll_col: str) -> float:
        is_pm25 = (poll_col == POLLUTANTS.get("PM2.5", "pm2.5"))
        if plot_type == "full_raw":
            return STYLE["ymax_full_raw_pm25"] if is_pm25 else STYLE["ymax_full_raw_pm10"]
        elif plot_type == "hourly":
            return STYLE["ymax_hourly_pm25"] if is_pm25 else STYLE["ymax_hourly_pm10"]
        elif plot_type == "halfhourly":
            return STYLE["ymax_halfhourly_pm25"] if is_pm25 else STYLE["ymax_halfhourly_pm10"]
        return 400

    total = 0

    for poll_label, poll_col in POLLUTANTS.items():
        poll_tag = poll_label.replace(".", "")   # PM25 / PM10

        for series_tag, (dfs, sheets) in series_map.items():
            base = os.path.join(OUTPUT_DIR, series_tag + "_series")

            for pt_key, (res_rule, subfolder, title_sfx) in plot_types.items():
                ymax  = get_ymax(pt_key, poll_col)
                fname = f"{series_tag}_{poll_tag}_daytype_{pt_key}.jpg"
                out   = os.path.join(base, subfolder, fname)
                title = (f"{series_tag}-series  |  {poll_label}  |  {title_sfx}  "
                         f"|  Day-type comparison")

                print(f"[{pt_key}] {series_tag} | {poll_label}")
                plot_daytype(dfs, sheets, poll_col, poll_label,
                             res_rule, ymax, out, plot_title=title)
                total += 1

    print(f"\n✅  Done — {total} plots saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Loading data …

── PARSE DIAGNOSTICS ──────────────────────────────────────────────────
  [S1_Dharavi]
    timestamp[0] raw  : '2026-03-12T04:30:47.046Z'
    ts[0]        raw  : Timestamp('2026-03-12 09:58:48')
    plot_dt[0] parsed : 2026-03-12 00:00:28
    valid rows        : 47095 / 47095  |  failed: 0
    day-type counts   : {'Weekday': 23634, 'Friday': 7887, 'Sunday': 7863, 'Saturday': 7711}
  [S2_AntopHill]
    timestamp[0] raw  : '2026-03-05T07:15:13.701Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:41:43')
    plot_dt[0] parsed : 2026-03-05 00:00:26
    valid rows        : 65116 / 65116  |  failed: 0
    day-type counts   : {'Weekday': 33774, 'Sunday': 10450, 'Saturday': 10448, 'Friday': 10444}
  [S3_CAAQMS]
    timestamp[0] raw  : '2026-03-05T07:15:09.425Z'
    ts[0]        raw  : Timestamp('2026-03-05 12:43:23')
    plot_dt[0] parsed : 2026-03-05 00:00:13
    valid rows        : 53716 / 53716  |  failed: 0
    day-type counts   : {'Weekday': 26388, 'Sunday': 10327, 'Satu